# Phase 7 — Block design (steps 72–74)

Phases 1–6 explained *why* each block exists. Phase 7 designs them — and the first design step is to decide how much each block is allowed to cost: jitter, power and area (step 72); the second is the loop itself (step 73). This notebook splits the proposal's targets (1.4 ps typ / 3.2 ps max with FFNC, −110/−120 dBc/Hz at 300 kHz, 9 mW, 0.08 mm²) into one allocation per block, shows which allocations are critical, and lets you move them with sliders. The same numbers are in `docs/design/step72_budget.md`; the model is `design/budget/step72_budget.py`.

**How to use this notebook.** Every step follows the same loop: **What happens** (the observation) →
**How it happens** (the mechanism and the equations, in Razavi's notation) → **Why it happens** (the physical
reason) → an interactive figure → **Try it** (what to change and what to look for) → **Interview questions**
(click a question to open the answer; try to answer it aloud first).

Notation (Razavi, *RF Microelectronics* ch. 9 and *Design of CMOS PLLs*): $\phi_{in}$, $\phi_{out}$ input and output
phase; $\omega_{out} = \omega_0 + K_{VCO} V_{cont}$; $K_{PD}$ phase-detector gain; $I_p$ charge-pump current;
$R_1, C_1, C_2$ loop filter; $N$ divide ratio; $\omega_n$, $\zeta$ natural frequency and damping;
$H(s) = \Phi_{out}/\Phi_{in}$. Frequencies $f$ in Hz, angular $\omega = 2\pi f$ in rad/s.
Design numbers are the submitted proposal's: $f_{REF} = 50$ MHz, $N = 32$, $f_{out} = 1.6$ GHz, $f_c \approx 2.5$ MHz.

## Contents

- [Step 72 — Phase-noise and power budget per block, and the jitter target](#step-72)
- [Step 73 — Loop parameters (f_c, PM, β, R1, C1, C2) from the out2 jitter model](#step-73)
- [Step 74 — VCO redesign: stages, LVT, sizing for frequency, K_VCO and phase noise](#step-74)

*3 steps, 109 interview questions.*

In [ ]:
# setup - run this cell first (Shift+Enter). Figures are interactive: drag the slider under each plot,
# hover for values, drag to zoom, double-click to reset.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sspll_learn as sl
from sspll_learn import P, show, sweep, bode, kT
np.set_printoptions(precision=4, suppress=True)
print("ready - proposal design point:", f"{P['fref']/1e6:.0f} MHz x {P['N']} = {P['f0']/1e9:.2f} GHz")

<a id='step-72'></a>
## Step 72 — Phase-noise and power budget per block, and the jitter target

*Roadmap step 72: budget table*

### What happens
Every block — crystal, reference receiver, SSPD, charge pump, loop-filter resistor, ring, current source, NCB, LDO — adds its
own small random timing error to the output edges. These errors are independent, so the output jitter is their
**root-sum-square**, each shaped by the path from the block to the output. A budget fixes, *before* any transistor is sized,
how much each block may contribute so the sum meets the specification, and gives each block a pass/fail number in its own
units.

### How it happens
Each source enters at a physical place in the loop; its transfer to the output follows from Phases 3–6:

$$S_{out}(f) = \big[N^2 S_{REF} + S_{PD}\big]\big|H + G_c(1-H)\big|^2 + S_{VCO}\big|(1-G_c)(1-H)\big|^2
 + S_{NCB} + (\omega_0 K_s)^2 S_{v,LDO} + |G_c|^2 S_{alias}$$

$$\sigma_t = \frac{1}{2\pi f_0}\sqrt{\int_{10\,\mathrm{kHz}}^{40\,\mathrm{MHz}} S_{out}(f)\,df},\qquad
\sigma_t^2 = \sum_i \sigma_{t,i}^2 .$$

Each block's physics gives its source in its own units, then converted to dBc/Hz:

| Block | Source | Conversion |
|---|---|---|
| REF (XO + receiver) | floor $L_{REF}$ | $\times N^2$ (+30.1 dB); per-edge jitter $\sigma = \sqrt{L f_{REF}}/(2\pi f_{REF})$ |
| SSPD | $kT/C$ | $L = kT/(2 C_s A^2 f_{REF})$ |
| Pulsed-Gm CP | $4kT\gamma g_m$ during duty $d$ | $L = kT\gamma/(2A^2 g_m d)$, with $\beta = 2A g_m d$ |
| Loop-filter $R_1$ | $4kTR_1$ | $\times |C_1/(C_1+C_2+sR_1C_1C_2)|^2\,(K_{VCO}/\omega)^2$ |
| CCO | $i_n$ (A/√Hz) | $S_\varphi = (K_I i_n)^2/f^2$ |
| NCB / LDO | floor / $v_n$ | 1:1 / $(\omega_0 K_s v_n)^2$ |

### Why it happens this way
The reference is a *time* error, so it is multiplied by $N^2$; the SSPD and CP see the VCO directly, so they are not. The
oscillator's noise is high-passed by the loop $(1-H)$ and again by $(1-G_c)$ with FFNC. The NCB and LDO come *after*
everything, so nothing filters them. Aliasing is the price of FFNC: the SSPD folds VCO noise at $f \pm kf_{REF}$ into the band
and the cancellation path adds it to the output. The relative size of the contributions below follows from these facts.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

rows = {}
for c in ("typ", "pess"):
    for on in (False, True):
        rows[(c, on)] = total(c, on)[0]
print(f"{'block':40s} {'typ off':>8s} {'typ on':>8s} {'pess off':>9s} {'pess on':>8s}   (ps rms, 10 kHz-40 MHz)")
for k in rows[("typ", True)]:
    v = [rows[(c, o)].get(k, 0)*1e12 for c in ("typ", "pess") for o in (False, True)]
    print(f"{k:40s} {v[0]:8.3f} {v[1]:8.3f} {v[2]:9.3f} {v[3]:8.3f}")
print(f"{'spec':40s} {'2.3 typ':>8s} {'1.4 typ':>8s} {'4.7 max':>9s} {'3.2 max':>8s}")

**Figure 72.1 — where the jitter comes from** (typ, FFNC off vs on). Hover for values.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

j0, _, _ = total("typ", False); j1, _, _ = total("typ", True)
names = sorted([k for k in j1 if k != "TOTAL"], key=lambda k: j1[k])
fig = go.Figure()
fig.add_trace(go.Bar(y=names, x=[j0.get(k, 0)*1e12 for k in names], orientation="h", name=f"FFNC off ({j0['TOTAL']*1e12:.2f} ps)", marker_color="#2a78d6"))
fig.add_trace(go.Bar(y=names, x=[j1[k]*1e12 for k in names], orientation="h", name=f"FFNC on ({j1['TOTAL']*1e12:.2f} ps)", marker_color="#eb6834"))
fig.update_layout(barmode="group", height=480, xaxis_title="rms jitter contribution (ps)", title="Jitter budget per block, typ")
show(fig)

**Try it.** Square the bars: the ring's $1.87^2 = 3.5$ ps² is 78% of the FFNC-off variance; the SSPD's $0.014^2$ is 0.004%.
A block at one third of the largest adds only 5% to the total — that is why you do not spend effort on small bars.

**Figure 72.2 — the ring 1/f³ corner and the 300 kHz spec.** The proposal specifies −110 dBc/Hz typ at 300 kHz with FFNC off.
Move the ring's 1/f³ corner: at the simulated 1.1 MHz the spec is missed by ~1.5 dB; the budget allocates ≤ 0.6 MHz.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

def frame(fc3):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total("typ", on, over={"ring VCO core": (-98.35, fc3)})
        tr.append(dict(x=f, y=L, name=f"FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'}: {j['TOTAL']*1e12:.2f} ps, L(300k) {L[i300]:.1f}")
    tr.append(dict(x=[3e5, 3e5], y=[-110, -120], name="spec typ (off, on)", mode="markers",
                   marker=dict(size=9, color="#59636e", symbol="circle")))
    return tr, f"ring f_c3 = {fc3/1e6:.2f} MHz — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) at 1.6 GHz (dBc/Hz)", range=[-140, -95])
show(sweep([0.2e6, 0.3e6, 0.4e6, 0.5e6, 0.6e6, 0.8e6, 1.1e6, 1.5e6, 2.2e6], frame, "ring 1/f³ corner",
           fmt="{:.2g} Hz", active=4, layout=lay))

**Figure 72.3 — loop bandwidth as the fallback knob.** If step 74 cannot lower the corner, `icp_trim` raises $f_c$ (it scales
$\beta$). Watch FFNC-off improve while FFNC-on barely moves (the cancellation already removes the VCO noise near $f_c$), and
remember the price: the sample-and-hold delay costs $\approx 180°\,f_c/f_{REF}$ of phase margin.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

def frame(fc):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total("typ", on, fc=fc)
        tr.append(dict(x=f, y=L, name=f"FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'} {j['TOTAL']*1e12:.2f} ps, L300k {L[i300]:.1f}")
    return tr, f"f_c = {fc/1e6:.1f} MHz (hold-delay PM loss {180*fc/50e6:.0f}°) — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) (dBc/Hz)", range=[-140, -95])
show(sweep([1.5e6, 2.0e6, 2.5e6, 2.8e6, 3.0e6, 3.5e6, 4.0e6], frame, "loop bandwidth", fmt="{:.2g} Hz", active=2, layout=lay))

**Figure 72.4 — the CCO current noise.** The CCO's thermal floor is fixed by physics: a source of current $I$ with headroom
$V_h$ has $i_n^2 \approx 4kT\cdot 2I/V_h$ → about 7 pA/√Hz at 1.31 mA and 1.2 V. Only its flicker corner is a design choice
(large-area devices, RC-filtered mirror gate). Slide the corner.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

KI = 1.22e12
def frame(fcf):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, S = total("typ", on, over={"CCO current DAC + mirror": (-104.2, fcf)})
        tr.append(dict(x=f, y=10*np.log10(S["CCO current DAC + mirror"]/2), name=f"CCO part, FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'} total {j['TOTAL']*1e12:.2f} ps (CCO {j['CCO current DAC + mirror']*1e12:.2f})")
    return tr, f"CCO 1/f corner {fcf/1e6:.2f} MHz, i_n = 7.2 pA/√Hz at 1 MHz — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="CCO contribution at the output (dBc/Hz)", range=[-160, -100])
show(sweep([0.05e6, 0.1e6, 0.2e6, 0.3e6, 0.5e6, 1.1e6, 2.2e6], frame, "CCO flicker corner", fmt="{:.2g} Hz", active=3, layout=lay))

**Figure 72.5 — sensitivity.** Total jitter when one block alone is 3 dB worse than its allocation. Long bars = critical allocations; short bars = slack you can trade for power, area or robustness.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

sens = bud.sensitivity()
base = total("typ", True)[0]["TOTAL"]*1e12
names = sorted(sens, key=lambda k: sens[k][1])
fig = go.Figure(go.Bar(y=names, x=[sens[k][1] - base for k in names], orientation="h", marker_color="#eb6834",
                       name="FFNC on", hovertemplate="%{y}: +%{x:.3f} ps<extra></extra>"))
fig.update_layout(height=430, xaxis_title=f"increase of total typ jitter (ps) over {base:.2f} ps", title="Block +3 dB → total jitter (FFNC on)")
show(fig)

### Power and area

Blocks on the 1.8 V LDO rail draw their current from VDDA = 3.3 V, so every mA there costs **3.3 mW**; the ring is fed from
VDDA through the CCO (4.86 mW). Logic that can run from VDDD costs 1.8 mW/mA — that is why dividers and FLL logic live there.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

tot = {"on": 0, "off": 0, "on_w": 0, "off_w": 0}
print(f"{'block':46s} {'supply':6s} {'I (mA)':>7s} {'P typ':>7s} {'P worst':>8s}")
for name, sup, i, p, pw, ncb in bud.power_table():
    print(f"{name:46s} {sup:6s} {i:7.3f} {p:7.2f} {pw:8.2f}")
    tot["on"] += p; tot["on_w"] += pw
    if not ncb: tot["off"] += p; tot["off_w"] += pw
print(f"TOTAL FFNC on {tot['on']:.2f} mW (worst {tot['on_w']:.2f}; spec 9.0 typ / 15 max)")
print(f"TOTAL FFNC off {tot['off']:.2f} mW (worst {tot['off_w']:.2f}; spec 7.5 typ / 13 max)")
j = total("typ", True)[0]["TOTAL"]
print(f"FoM = {10*np.log10(j**2*tot['on']):.1f} dB (proposal -227.5)")
print("\narea:", ", ".join(f"{n} {a}" for n, a in bud.AREA), f"\n-> {sum(a for _, a in bud.AREA)} um^2 (spec 80,000 typ)")

### Layout convenience, decided now

The budget tells you which nodes are noise-critical, so the schematic can already be drawn the way it will be laid out:

- **Ring (dominant):** 5 identical cells, U-shaped placement so the wrap-around wire matches the others, dummy loads on unused
  taps, integer fingers per device so each cell is one abutting row.
- **CCO (second):** 1:8 mirror as 9 unit devices in a common-centroid row with end dummies; binary DAC from unit cells;
  RC-filtered gate (long high-resistance poly + MOS cap under the array); 3.3 V devices in their own guard ring.
- **Reference receiver (×N²):** short, shielded route from the pad; own VDDA branch; far from the ring.
- **SSPD, CP:** fully differential, mirror-symmetric, unit MIM tiles with a dummy ring, ABBA input pair, non-minimum L.
- **Loop filter:** R1 as equal legs of wide poly; C1 MOS cap with MIM stacked above (~9.5 fF/µm², V_cont > 0.6 V); C2 linear MIM; tiles ≤ 30×30 µm.
- **NCB / LDO:** NCB at the output buffer with its own LDO decap; LDO pass device as unit fingers along the supply edge.
- **Everywhere:** few distinct unit devices; supply domains as bands (VDDA / LDO 1.8 V / VDDD); level shifters only at band edges.

### Interview questions — step 72 (42)

<details><summary><b>Q1.</b> What is a noise budget, and why do it before designing any transistor?</summary>

A table that splits the system target (here 1.4 ps typ / 3.2 ps max rms jitter, −110/−120 dBc/Hz at 300 kHz) into one allocation per block, each in that block's own units (pA/√Hz, nV/√Hz, fs per edge). It turns one system number into pass/fail tests a single block can be designed and simulated against, and it shows *early* which block dominates — so the design effort (and power and area) goes there and not to blocks that cannot matter.

</details>

<details><summary><b>Q2.</b> Why do jitter contributions add in power (rms²) and not linearly?</summary>

The sources are physically independent (different devices), so they are uncorrelated and their variances add: σ² = Σσᵢ². Consequence: a block at half the jitter of the largest one adds only 12% to the total; a block at 1/3 adds 5%. Fixing small contributors is wasted effort.

</details>

<details><summary><b>Q3.</b> Write the output phase-noise PSD of the SSPLL with FFNC in terms of the block sources.</summary>

S_out(f) = [S_REF·N² + S_PD,eq]·|H + G_c(1−H)|² + S_VCO,eq·|(1−G_c)(1−H)|² + S_NCB + S_alias·|G_c|², where H is the closed-loop transfer, G_c the cancellation path (SSPD gain × NCB gain, with sinc hold, half-period delay and pole) and the VCO-like sources (ring, CCO, loop-filter R) all enter at the VCO. Jitter = √(∫S_out df)/(2πf_out), here 10 kHz–40 MHz.

</details>

<details><summary><b>Q4.</b> Why is the reference noise multiplied by N² but the SSPD noise is not?</summary>

The reference sets the *time* of each edge; a time error Δt at f_REF becomes a phase error 2πf_out·Δt at the output, i.e. N times the reference phase error → N² in power. The SSPD compares the VCO directly (no divider), so its voltage noise divided by its gain is already a VCO-phase error: no N. That is the whole point of sub-sampling (−130 vs −110.9 dBc/Hz detector floor at the output, step 8).

</details>

<details><summary><b>Q5.</b> Why does FFNC make the reference contribution *worse* (0.20 → 0.31 ps)?</summary>

With FFNC on, the in-band transfer becomes |H + G_c(1−H)|²: above the loop bandwidth the cancellation path also passes the SSPD's view of the reference, so reference (and SSPD/CP) noise reaches the output up to ≈f_REF/8 instead of f_c. FFNC trades VCO noise for reference noise; it wins because the VCO term is 10× larger.

</details>

<details><summary><b>Q6.</b> Which block dominates the budget, and what does that tell you?</summary>

The ring VCO core (1.87 ps off / 1.13 ps on typ), then the CCO (0.90 / 0.56). Everything else together is < 0.5 ps. So power, area and design time go to the oscillator and its current source; the SSPD, CP and NCB can be designed for linearity, spurs and robustness rather than noise.

</details>

<details><summary><b>Q7.</b> The SSPD kT/C noise contributes only 22 fs. Why not shrink the sampling capacitor to save power?</summary>

Because the cap also sets the charge-sharing/kick-back spur and the hold droop; and the SSPD noise is in-band where FFNC passes it. 100 fF keeps kT/C at −147.8 dBc/Hz (17 dB below the CP allocation), so it is not the limit; going to 25 fF costs +6 dB noise but buys little power (the ring buffers that drive it set the power). The budget says: free to choose the cap for spur and matching, as long as Cs ≥ ~50 fF per side.

</details>

<details><summary><b>Q8.</b> Derive the SSPD kT/C phase-noise floor.</summary>

Each side samples kT/C; the differential sample has v² = 2kT/C. The differential slope is 2A per radian of VCO phase, so one sample carries φ² = (2kT/C)/(2A)² = kT/(2CA²). Samples are independent, so this variance is white over 0…f_REF/2: S_φ = φ²/(f_REF/2) = kT/(C·A²·f_REF), and L = S_φ/2 = kT/(2·C·A²·f_REF) = −147.8 dBc/Hz for C = 100 fF, A = 0.5 V, f_REF = 50 MHz.

</details>

<details><summary><b>Q9.</b> How is the pulsed-Gm charge-pump noise budgeted, and why does the duty cycle matter?</summary>

The transconductor's output noise current 4kTγg_m only reaches the loop filter during the pulse (duty d ≈ 1%, 200 ps of 20 ns). The useful signal current is β = 2A·g_m·d (A per rad). Referred to VCO phase: S_φ = 4kTγg_m·d/β² → L ≈ kTγ/(2A²g_m d) ≈ −140.8 dBc/Hz white, plus a 1/f corner allocated ≤ 300 kHz. Shorter pulses lower the signal more than the noise at equal g_m, so d is set by β and the spur, not by noise.

</details>

<details><summary><b>Q10.</b> Why is the loop-filter resistor a visible contributor (0.28 ps) and what are the knobs?</summary>

Its 4kTR voltage noise (12.6 nV/√Hz for 9.61 kΩ) is filtered by C1/(C1+C2+sRC1C2) and integrated by K_VCO/s, so it behaves like VCO noise (high-passed by 1−H, cancelled by FFNC). Knobs: larger β (more CP current → smaller R, larger C, more area), smaller K_VCO (needs a wider coarse range from the CCO DAC), or a dual-path filter. We accept it: 0.28 ps adds only ~1% to a 2.1 ps total.

</details>

<details><summary><b>Q11.</b> Why must the ring 1/f³ corner be halved (1.1 → 0.6 MHz) in this budget?</summary>

With the simulated 1.1 MHz corner the proposal's own model gives L(300 kHz) = −108.1 dBc/Hz FFNC-off typ, 1.9 dB short of the −110 typ in the specification. At 300 kHz the ring is in its 1/f³ region, so L ∝ (1 + f_c3/f); halving f_c3 plus a lower-flicker CCO brings it to −110.2. Halving needs ~2× gate area in the ring transistors that upconvert flicker, or better rise/fall symmetry (smaller DC term of the ISF, c₀).

</details>

<details><summary><b>Q12.</b> What determines the 1/f³ corner of a ring oscillator?</summary>

Hajimiri–Lee: f_1/f³ = f_1/f(device) · (c₀/c_rms)². c₀ is the DC value of the ISF (zero for a perfectly symmetric waveform), c_rms its rms. So the corner falls with device area (lower device flicker corner) and with symmetry of rise and fall edges (pMOS/nMOS strength balance, equal loading).

</details>

<details><summary><b>Q13.</b> Why is the CCO noise allocated in pA/√Hz, and how do you convert it?</summary>

The CCO is a current source: what it delivers to the ring is a noisy current. Frequency noise = K_I·i_n (K_I = ∂f/∂I = 1.22×10¹² Hz/A), phase noise S_φ = (K_I i_n)²/f², L = S_φ/2. L(1 MHz) = −104.2 dBc/Hz ↔ i_n = 7.2 pA/√Hz.

</details>

<details><summary><b>Q14.</b> Why can't the CCO noise be made arbitrarily small?</summary>

A MOS current source of current I with overdrive (headroom) V_ov has i_n² ≈ 4kTγ·2I/V_ov. At 1.31 mA and ~1.2 V of headroom left by VDDA_min 2.97 V minus the ~1.6–1.8 V ring node, the thermal floor is ≈6–7 pA/√Hz. Degeneration gives the same limit (4kT/R_s with R_s = V_h/I). Only more headroom or less current helps — the budget therefore allocates the thermal floor and attacks only the flicker part.

</details>

<details><summary><b>Q15.</b> Why does a filtered mirror gate help the CCO noise?</summary>

In a 1:8 mirror the reference-side noise (DAC unit cells, IBIAS mirror, diode device) is multiplied by 8 at the output. An RC low-pass on the shared gate (corner ≪ 10 kHz, e.g. a long poly resistor and a MOS cap) removes it inside the integration band, so only the output devices' own noise remains. It costs area, not power.

</details>

<details><summary><b>Q16.</b> Why is the NCB intrinsic noise budget so small (31 fs) yet still important?</summary>

It is added *after* the loop and the cancellation: nothing filters it, so it appears 1:1 at the output with FFNC on, all the way to 40 MHz. −150 dBc/Hz white with a 1 MHz flicker corner gives 31 fs. If the delay line were a long chain of starved inverters, its noise would grow with its delay and could reach hundreds of fs — so the NCB is kept short (2 cells, small delay range).

</details>

<details><summary><b>Q17.</b> How does LDO noise get into the output, and how is that budgeted?</summary>

The NCB delay depends on its supply (≈104 ps/V for 4 inverter edges at ~26 ps/V each): a supply noise density v_n becomes a timing noise K_s·v_n, i.e. phase 2πf_out·K_s·v_n. For ≤ 26 fs contribution the LDO output noise must be ≤ 40 nV/√Hz (10 kHz flicker corner). The ring itself is fed by the high-impedance CCO, not the LDO, so LDO noise does not modulate the oscillator directly.

</details>

<details><summary><b>Q18.</b> What is the aliasing term in the FFNC budget?</summary>

The SSPD samples at f_REF, so VCO noise at f ± k·f_REF folds into the 0…f_REF/2 band (Thijssen: S_s(f) = f_REF Σ S_φ(f − k f_REF)). The cancellation path then *adds* this folded noise to the output (it is not correlated with the output's own noise at f). It costs 0.19 ps typ — the main reason FFNC cannot cancel everything.

</details>

<details><summary><b>Q19.</b> Why is the jitter integrated from 10 kHz to 40 MHz?</summary>

The proposal's specification band. The lower edge excludes slow wander a downstream system tracks; the upper edge is the Nyquist range of a 100 MHz-class sampling clock divided output and where the ring's −20 dB/dec skirt has fallen well below the in-band level. A budget must use the same band as the spec.

</details>

<details><summary><b>Q20.</b> How do you check a budget against the reference range 45–55 MHz?</summary>

Re-run with f_REF = 45 and 55 MHz: f_out scales, the ring's L at 1 MHz scales ~ f_out (same power per edge), the aliasing and the cancellation bandwidth move with f_REF. Typ jitter goes 2.23/1.51 ps at 45 MHz to 2.01/1.23 ps at 55 MHz (off/on) — all inside the maxima.

</details>

<details><summary><b>Q21.</b> What does the 'pessimistic' column represent?</summary>

A coherent worst plausible set for the noise numbers: ring +1 dB and 2× 1/f³ corner, CCO +3 dB and 2× corner, reference receiver +4 dB with a 100 kHz corner, SSPD amplitude 0.35 V, γ = 1.5, NCB −146 dBc/Hz, LDO 80 nV/√Hz and a 30% FFNC gain error. It is not a SPICE corner; it is the budget's stress case that must still meet the *maximum* spec.

</details>

<details><summary><b>Q22.</b> The FFNC-on L(300 kHz) is −119.9 typ against a −120 typ spec. Is that a fail?</summary>

It is a typ number at the limit, not a guaranteed limit (the max is −110, with 4.5 dB margin pess). At 300 kHz with FFNC on the floor is set by the reference ×N²: crystal −155 and receiver −155 → −121.2 dBc/Hz together, before any on-chip noise. Only a cleaner receiver (−157 for ~+0.1 mW) or a better board crystal moves it. This is recorded as a finding.

</details>

<details><summary><b>Q23.</b> How is the power budget built and why does the LDO domain cost more?</summary>

Every block gets a current on a named supply; power = I × supply voltage at the slot. Blocks on the 1.8 V LDO rail draw their current from VDDA = 3.3 V, so each mA costs 3.3 mW, not 1.8 mW: the LDO burns the 1.5 V difference. That is why logic that can live on VDDD (1.8 V, 1.8 mW/mA) is put there.

</details>

<details><summary><b>Q24.</b> Why is the ring fed from VDDA through a CCO instead of from the LDO?</summary>

A current-fed ring has low supply sensitivity (the CCO's output impedance isolates it) and the current DAC also gives the coarse tuning. The price: the ring burns its current at 3.3 V (4.86 mW). Running it from the LDO would cost the same power from VDDA and add the LDO noise directly to the oscillator.

</details>

<details><summary><b>Q25.</b> What is the worst-case power and how is it estimated?</summary>

Ring at 1.76 GHz, ss, 110 °C, VDDA 3.63 V: 7.46 mW (from the proposal's simulated basis). Other blocks: typ current × 1.3 at the maximum supply. Total 13.5 mW FFNC on / 11.4 mW off against 15/13 mW maximum.

</details>

<details><summary><b>Q26.</b> What is the jitter figure of merit and what does this budget give?</summary>

FoM = 10·log₁₀[(σ/1 s)²·(P/1 mW)]. σ = 1.36 ps, P = 9.08 mW → −227.8 dB (proposal −227.5). Note the FoM rewards power and jitter equally in dB: halving the jitter (−6 dB) is worth 4× the power.

</details>

<details><summary><b>Q27.</b> Why does the area table now include the LDO?</summary>

Decision D4 (step 8) added an on-chip LDO for the clean 1.8 V rail; the proposal's area basis listed every block except it. ~4000 µm² (pass device for ~1.2 mA, error amp, compensation and output cap partly as MOS cap). Total 57,500 µm² — still below the 0.08 mm² typ.

</details>

<details><summary><b>Q28.</b> How would you decide between spending 1 mW on a cleaner VCO and 1 mW on a better NCB?</summary>

Use the sensitivity of the total to each block: ring +3 dB moves the typ total from 1.36 to 1.77 ps; NCB +3 dB changes nothing visible. The marginal improvement per mW is far larger at the VCO. A budget makes that comparison numeric.

</details>

<details><summary><b>Q29.</b> What is the 'sensitivity' table and how is it used during design?</summary>

For each block, the total jitter when that block alone is 3 dB worse than allocated. It shows which allocations are critical (ring, CCO, reference) and which have slack (SSPD, CP, NCB, LDO). Slack can be traded for power, area or robustness during the block designs.

</details>

<details><summary><b>Q30.</b> How will each allocation be verified in Phase 8?</summary>

Each block's testbench measures its allocation in its own units: SSPD/CP with .noise and PSS-equivalent transient noise (tb_sspd_char, tb_sscp_*), ring free-running phase noise by transient noise and ISF (tb_vco_*), CCO output current noise by .noise (tb_cco_ac), LDO output noise and PSRR (tb_ldo_ac), NCB delay noise and supply sensitivity (tb_ncb_*). The results replace the allocations in step72_budget.py, which recomputes the total.

</details>

<details><summary><b>Q31.</b> Why keep the budget in a script instead of a spreadsheet?</summary>

Because the transfer functions are frequency-dependent and complex (H, 1−H, G_c, aliasing) — a spreadsheet of single numbers cannot integrate them. A script also re-runs automatically when a simulated curve replaces an allocation, and it is version-controlled next to the schematics.

</details>

<details><summary><b>Q32.</b> What layout decisions follow directly from the noise budget?</summary>

(1) The ring and its CCO get the cleanest, symmetric placement (dominant noise); (2) the CCO output mirror is large-area, common-centroid, with a filtered gate; (3) the reference receiver path is short and shielded (it is ×N²); (4) the SSPD caps are placed symmetrically with dummies (spur, not noise); (5) the NCB is next to the output with its own LDO decap; (6) the loop-filter C1 stacks MIM over MOS to fit the area.

</details>

<details><summary><b>Q33.</b> Why is the loop-filter C1 a MOS capacitor with MIM on top, and what is the catch?</summary>

C1 = 18.2 pF would need ~9100 µm² as MIM alone (~2 fF/µm²), more than the CP+LF area allocation. A thin-oxide MOS cap (7.5 fF/µm² at 0.9 V) with a MIM cap stacked above gives ~9.5 fF/µm² → ~1,900 µm² (~11.5 fF/µm² with both MIM layers). Catch: a MOS cap is non-linear and must be kept in strong inversion/accumulation across the tuning range — acceptable for C1 (the large zero cap), not for C2.

</details>

<details><summary><b>Q34.</b> What is the effect of a loop-gain (K_VCO or β) error on the budget?</summary>

f_c moves (f_c ∝ βK_VCO): a lower f_c lets more VCO noise through and raises FFNC-off jitter; a higher f_c helps here (2.8 MHz closes the 300 kHz spec alone) but costs phase margin through the sample-and-hold delay (≈ 180°·f_c/f_REF). icp_trim[1:0] is the knob to recentre f_c after silicon or after the K_VCO of step 74 is known.

</details>

<details><summary><b>Q35.</b> Why 2× margin between pessimistic budget and the max spec?</summary>

The budget's models (ISF-based ring noise, γ, flicker coefficients) carry several-dB uncertainty, and layout parasitics add noise and power. The proposal says maximum values keep about twice the pre-layout margin; here pess 2.71/1.73 ps vs 4.7/3.2 ps max is 1.7–1.8×.

</details>

<details><summary><b>Q36.</b> Explain 'L = S_φ/2' and why the code uses 2·10^(L/10).</summary>

L(f) is defined as single-sideband noise relative to the carrier; for small phase noise L(f) = S_φ(f)/2 where S_φ is the one-sided phase PSD. Integrating jitter needs S_φ = 2·10^(L/10) rad²/Hz, then σ_t = √∫S_φ df / (2πf_0).

</details>

<details><summary><b>Q37.</b> How does a flicker (1/f) noise source in the CP or SSPD show up at the output?</summary>

Below the loop bandwidth H ≈ 1, so a 1/f input source appears as 1/f at the output — it raises L(f) at low offsets (10–100 kHz). That is why the CP/SSPD allocations include a 300 kHz flicker corner and why their devices are not minimum length.

</details>

<details><summary><b>Q38.</b> What would happen to the budget if the reference were 25 MHz with N = 64?</summary>

The reference term grows by 20·log(2) = 6 dB (N²), the SSPD term grows because A²·f_REF halves (+3 dB), the cancellation bandwidth halves (f_REF/8) and aliasing worsens — the 50 MHz reference is a deliberate budget choice.

</details>

<details><summary><b>Q39.</b> Which allocations are measured on silicon, and how?</summary>

Total jitter and L(f) on CLK_OUT (/16 + 24.1 dB), FFNC on vs off on the same chip, supply sensitivity with the VTEST tone, and power per domain from slot currents with blocks enabled in turn. Individual block noise is not directly observable — the budget lets you attribute the measured total.

</details>

<details><summary><b>Q40.</b> A colleague proposes a 3-stage ring to save power. What does the budget say?</summary>

The step-6 study: 3 × 1.0 at 0.61 mA gives −87.4 dBc/Hz vs −94.2 for 5 × 1.5 — 6.8 dB worse at the dominant block; jitter would roughly double. Saving ~2 mW does not pay; the 5-stage verdict stands.

</details>

<details><summary><b>Q41.</b> Why is the FLL charge pump nearly absent from the noise budget?</summary>

It has a dead zone: in lock the phase error sits inside it and the FLL CP is off — no noise current reaches the loop filter. Its leakage and switch charge injection matter for spurs (step 73), not for random jitter.

</details>

<details><summary><b>Q42.</b> What are the three numbers every block designer must take away from step 72?</summary>

Its noise allocation (in its own units), its supply current on a named supply, and its area. Plus its layout rules from the budget: which nodes are noise-critical, which must be symmetric and which supply domain it lives in.

</details>

<a id='step-73'></a>
## Step 73 — Loop parameters (f_c, PM, β, R1, C1, C2) from the out2 jitter model

*Roadmap step 73: design sheet*

### What happens
The loop filter and the charge-pump gain decide how the work is split: **below $f_c$** the loop makes the VCO follow the
clean reference, **above $f_c$** the VCO runs free and FFNC removes its noise up to about $f_{REF}/8$. Step 73 picks
$f_c$ and PM from the jitter model, computes $R_1, C_1, C_2$ for the *real, sampled* loop, and turns them into PDK devices
in whole layout tiles.

### How it happens
$$L(s) \approx \beta K_{VCO}\frac{Z(s)}{s}e^{-st_p},\quad Z(s) = \frac{1+sR_1C_1}{s(C_1+C_2)(1+sR_1\frac{C_1C_2}{C_1+C_2})},
\quad \beta = 2A\,g_m\frac{\tau}{T}$$

Closed-form start (zero and pole around $\omega_c$, spread $b=\tan(45°+\varphi_{lead}/2)$):
$C_{tot} = b\beta K_{VCO}/\omega_c^2$, $C_2 = C_{tot}/b^2$, $C_1 = C_{tot}-C_2$, $R_1 = b/(\omega_c C_1)$, with
$\varphi_{lead} = PM + 360°f_c t_p$. Then the **exact sampled model** corrects it: the SSPD samples once per period and the
Gm delivers one packet $q[k] = \beta T\theta_e[k]$ a delay $t_p$ later, so
$$x[k+1] = e^{AT}x[k] + e^{A(T-t_p)}B\,q[k],\qquad L(z) = \beta T\,C(zI-e^{AT})^{-1}e^{A(T-t_p)}B .$$

### Why it happens this way
Sampling and the packet delay cost phase that a continuous model does not see (it would give 40.6° instead of 50° at
$t_p$ = 11 ns). The loop sees only the product $\beta K_{VCO}$, so the big passives can be frozen now and the unknown
$K_{VCO}$ absorbed later by the small $g_m$ cells. A MOS capacitor is non-linear, so $C_1$ can be a dense MOS cap only
if $V_{cont}$ stays where it is flat.

**Figure 73.1 — what pins $f_c$.** Slide $f_c$ (PM 50°). Watch three numbers in the title: out2 jitter (barely moves),
out1 $L$(300 kHz) (needs $f_c \ge 2.5$ MHz for −110 dBc/Hz) and the FFNC jitter reduction (needs $f_c \le 2.5$ MHz for
35 %). Only 2.5 MHz meets both typ values.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

def frame(fc):
    tr = []; jj = {}
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total("typ", on, fc=fc)
        jj[on] = (j["TOTAL"]*1e12, L[i300])
        tr.append(dict(x=f, y=L, name=f"{'out2 (FFNC on)' if on else 'out1 (FFNC off)'}", line=dict(color=col)))
    red = 100*(1 - jj[True][0]/jj[False][0])
    return tr, (f"f_c {fc/1e6:.1f} MHz: out1 {jj[False][0]:.2f} ps, L300k {jj[False][1]:.1f} | out2 {jj[True][0]:.2f} ps | "
                f"FFNC reduction {red:.0f} %")
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) (dBc/Hz)", range=[-140, -95])
show(sweep([1.5e6, 2.0e6, 2.5e6, 3.0e6, 3.5e6, 4.0e6, 5.0e6], frame, "f_c", fmt="{:.2g} Hz", active=2, layout=lay))

**Figure 73.2 — the packet delay $t_p$ eats phase margin.** The filter is fixed (designed for $t_p$ = 2.5 ns). Slide
$t_p$ and watch the phase at crossover. The dashed line is the continuous model, which cannot see the delay at all.

In [ ]:
import sys, json, csv; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget"); sys.path.insert(0, "../design/loop")
import sampled_loop as sl
D = json.load(open("../design/loop/data/step73_design.json"))
R, C1, C2, KV, beta, tp, T = D["R"], D["C1"], D["C2"], D["KV"], D["beta"], D["tp"], 20e-9

f = np.logspace(4, np.log10(0.49/T), 400)
s = 2j*np.pi*f
Lc = beta*KV*(1 + s*R*C1)/(s*(C1 + C2)*(1 + s*R*C1*C2/(C1 + C2)))/s
def frame(t):
    L = sl.loop_gain(f, R, C1, C2, KV, beta, T, t)
    fc, pm_ = sl.margins(R, C1, C2, KV, beta, T, t)
    ph = np.degrees(np.unwrap(np.angle(L))); ph -= 360*np.round(ph[0]/360)
    return ([dict(x=f, y=ph, name="sampled loop ∠L", line=dict(color="#2a78d6")),
             dict(x=f, y=np.degrees(np.unwrap(np.angle(Lc))), name="continuous ∠L", line=dict(color="#eb6834", dash="dash"))],
            f"t_p = {t*1e9:.1f} ns: f_c = {fc/1e6:.2f} MHz, PM = {pm_:.1f}°  (delay lag at f_c = {360*fc*t:.1f}°)")
def lay(fig):
    fig.update_xaxes(type="log", title="frequency (Hz)"); fig.update_yaxes(title="phase of L (deg)", range=[-260, -100])
    fig.add_vline(x=D["fc"], line_width=1, line_color="#59636e")
show(sweep([0.5e-9, 1e-9, 2.5e-9, 5e-9, 8e-9, 11e-9], frame, "packet delay t_p", fmt="{:.2g} s", active=2, layout=lay))

**Figure 73.3 — loop-gain spread and `icp_trim`.** $m = (A g_m \tau K_{VCO})$/nominal varies over PVT. Slide $m$ and watch
the phase-step response (cycle by cycle, exact): small $m$ → slow and under-damped, large $m$ → fast but ringing. The
`icp_trim` codes multiply $\beta$ by 0.67 / 1 / 1.33 / 1.67 to pull $m$ back towards 1.

In [ ]:
import sys, json, csv; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget"); sys.path.insert(0, "../design/loop")
import sampled_loop as sl
D = json.load(open("../design/loop/data/step73_design.json"))
R, C1, C2, KV, beta, tp, T = D["R"], D["C1"], D["C2"], D["KV"], D["beta"], D["tp"], 20e-9

def frame(m):
    t, e, v = sl.step(R, C1, C2, KV, beta*m, T, tp, ncyc=150, phi_step=1.0)
    fc, pm_ = sl.margins(R, C1, C2, KV, beta*m, T, tp)
    return ([dict(x=t*1e6, y=e/(2*np.pi*1.6e9)*1e12, name="phase error", line=dict(color="#2a78d6", shape="hv")),
             dict(x=[0, 3], y=[10, 10], name="±10 ps", line=dict(color="#59636e", dash="dot")),
             dict(x=[0, 3], y=[-10, -10], name="−10 ps", line=dict(color="#59636e", dash="dot"), showlegend=False)],
            f"m = {m:.2f}: f_c = {fc/1e6:.2f} MHz, PM = {pm_:.1f}°")
def lay(fig):
    fig.update_xaxes(title="time after a 1 rad phase step (µs)", range=[0, 3]); fig.update_yaxes(title="phase error (ps)", range=[-60, 100])
show(sweep([0.35, 0.5, 0.67, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0], frame, "loop-gain multiplier m", fmt="{:.2f}", active=4, layout=lay))

**Figure 73.4 — why $V_{cont}$ must stay in 0.6–1.2 V.** The measured C-V of the nfet_01v8 MOS capacitor (ngspice, frozen
PDK) at three temperatures and three corners. Below about 0.6 V the channel disappears and $C_1$ collapses — the zero
would move up and the phase margin would fall.

In [ ]:
import csv
rows = list(csv.DictReader(open("../design/loop/data/moscap_cv.csv")))
fig = go.Figure()
cols = {"tt": "#2a78d6", "ss": "#eb6834", "ff": "#1baf7a"}; dash = {"-40": "dot", "27": "solid", "110": "dash"}
for c in ("tt", "ss", "ff"):
    for t in ("-40", "27", "110"):
        r = [x for x in rows if x["corner"] == c and x["temp"] == t]
        fig.add_trace(go.Scatter(x=[float(x["vg"]) for x in r], y=[float(x["c_mos_fF"])/100 for x in r], name=f"{c}, {t} °C",
                                 line=dict(color=cols[c], dash=dash[t], width=2 if t == "27" else 1.2)))
fig.add_vrect(x0=0.6, x1=1.2, fillcolor="#e6e8eb", opacity=0.5, line_width=0, annotation_text="V_cont window")
fig.update_layout(height=420, title="nfet_01v8 gate capacitance density (W = L = 10 µm)",
                  xaxis_title="V_gate (V)", yaxis_title="C (fF/µm²)")
show(fig)

### The design sheet

| Parameter | Value |
|---|---|
| $f_c$, PM | 2.50 MHz, 50.0° (exact sampled loop, packet delay $t_p$ = 2.5 ns, $\le$ 5 ns required) |
| $\beta$ | 0.848 µA/rad = $2A g_m d$ with $A$ = 0.5 V, $d$ = 1 %, $g_m$ = 84.8 µS = 3 units × 28.3 µS |
| $R_1$ | 11.01 kΩ — res_xhigh_po_2p85, L = 15.5 µm |
| $C_1$ | 17.11 pF — 2 tiles × (9 nfet_01v8 W = L = 10 µm + 1 cap_mim_m3_1 30×30 µm on top) |
| $C_2$ | 2.205 pF — 2 × cap_mim_m3_1 30 × 18.1 µm |
| zero / pole | 0.845 / 7.40 MHz; peaking 2.5 dB; $f_{3dB}$ 4.3 MHz; step settles in 0.34 µs |
| $V_{cont}$ window | 0.6–1.2 V |
| `icp_trim` | 00 → ×1, 01 → ×1.33, 10 → ×0.67, 11 → ×1.67 (Gm unit count 3 / 4 / 2 / 5) |
| residual spread after trim | $m$ within ×0.75–×1.3 → PM 45.5–50° at every R/C corner |

**Layout convenience built into these numbers:** $C_1$ is an integer number of identical tiles (β was rescaled to fit
the tiles, not the other way round); $C_2$ is two equal MIM rectangles; $R_1$ is one body of a legal width; the trim is
done with identical Gm unit cells, not with odd-sized devices; the MIM over the MOS cap shields the filter node from
routing above. Full sheet with every number: `docs/design/step73_loop.md`.

### Interview questions — step 73 (36)

<details><summary><b>Q1.</b> Why is the loop bandwidth chosen from the out2 (FFNC-on) jitter model and not from out1?</summary>

out2 is the output the specification is written for (1.4 ps typ). But out2 is almost flat in f_c (1.34–1.36 ps from 1.5 to 5 MHz) because FFNC cancels the VCO noise the loop would otherwise have to suppress. So out2 says 'anything in 1.5–5 MHz works', and the *other* specs pick the point inside that range.

</details>

<details><summary><b>Q2.</b> Then what actually pins f_c = 2.5 MHz?</summary>

Two specs pull in opposite directions. The FFNC-off phase noise at 300 kHz (−110 typ) needs f_c ≥ 2.5 MHz, because a wider loop suppresses more ring noise. The FFNC jitter reduction (35 % typ, 20 % min) needs f_c ≤ 2.5 MHz, because a wider loop leaves less VCO noise for FFNC to remove. 2.5 MHz is the only point that meets both typ values.

</details>

<details><summary><b>Q3.</b> Why does a wider loop reduce the benefit of FFNC?</summary>

FFNC removes VCO noise between f_c and about f_REF/8. If the loop already suppresses the VCO up to a higher f_c, the band where FFNC helps shrinks: at 4 MHz the reduction is 18 % (below the 20 % minimum), at 5 MHz only 9 %.

</details>

<details><summary><b>Q4.</b> Why 50° of phase margin and not 60°?</summary>

In the jitter model a higher PM lowers peaking and jitter slightly (1.36 → 1.30 ps at 60°), but the zero moves lower (the zero–pole spread b grows), which weakens low-frequency VCO suppression: L(300 kHz) FFNC-off goes from −110.2 to −108. 50° is the highest PM that still meets −110, and it keeps the overshoot (27 %) and peaking (2.5 dB) moderate.

</details>

<details><summary><b>Q5.</b> What is the 'sampled-data' loop and why is it needed here?</summary>

The SSPD looks at the phase only once per reference period and the pulsed Gm delivers one charge packet per period, a delay t_p after the sampling edge. A continuous-time model ignores both. The exact model propagates the filter + VCO state over one period with the matrix exponential, x[k+1] = e^{AT}x[k] + e^{A(T−t_p)}B·q[k], and evaluates L(e^{jωT}). It shows the continuous design would land at 48.7° (t_p = 0.5 ns) or 40.6° (t_p = 11 ns) instead of 50°.

</details>

<details><summary><b>Q6.</b> How large is the phase lag of the packet delay, and how do you compensate it?</summary>

A delay t_p adds 360°·f_c·t_p of lag at crossover: 2.3° at 2.5 ns, 9.9° at 11 ns. It is compensated by designing the filter for a larger lead (52.7° instead of 50° at t_p = 2.5 ns), i.e. a wider zero–pole spread, then iterating on the exact sampled model until f_c and PM land exactly.

</details>

<details><summary><b>Q7.</b> Why does t_p become a requirement on the clock generator?</summary>

With the filter fixed, PM falls from 51.3° (t_p = 0.5 ns) to 48.3° (5 ns) and 43.4° (11 ns). Firing the Gm pulse right after the hold transfer instead of half a period later is free in power and area, so step 77 must deliver t_p ≤ 5 ns (2.5 ns nominal).

</details>

<details><summary><b>Q8.</b> Write β for the pulsed-Gm charge pump and explain each term.</summary>

β = 2A·g_m·d (A per rad of VCO phase). 2A is the differential SSPD slope (V/rad) for a sampled amplitude A, g_m converts the held voltage into current, and d = τ/T is the fraction of the period the Gm conducts. The average current into the filter is β·θ_e. Here A = 0.5 V, d = 0.01 (200 ps / 20 ns), g_m = 84.8 µS → β = 0.848 µA/rad.

</details>

<details><summary><b>Q9.</b> Why can K_VCO still be unknown when the filter is frozen?</summary>

The loop depends only on the product β·K_VCO (C_tot = b·β·K_VCO/ω_c², R = b/(ω_c C1)). If step 74 lands a different K_VCO, g_m is rescaled (g_m ∝ 1/K_VCO) and the filter is unchanged. The large-area parts (C1, C2) are frozen now; the small-area part (Gm unit size) absorbs the uncertainty. That is a layout-driven decision.

</details>

<details><summary><b>Q10.</b> How were C1 and β picked together?</summary>

C1 was snapped to whole layout tiles first (2 tiles of 8.56 pF), then β was scaled by the same factor (×0.848) so β·K_VCO/C stays the same, and R (×1/0.848) and C2 (×0.848) followed. The loop is identical; the layout gets an integer number of identical tiles.

</details>

<details><summary><b>Q11.</b> What is in one C1 tile and why?</summary>

9 nfet_01v8 MOS-capacitor units (W = L = 10 µm, 3×3) with one 30×30 µm cap_mim_m3_1 above them: 6.73 pF + 1.82 pF = 8.56 pF at V_cont = 0.9 V in about 34×34 µm. The MOS cap gives density (7.5 fF/µm²), the MIM on top is free extra capacitance in the same footprint and shields the gate from routing above.

</details>

<details><summary><b>Q12.</b> What is the catch with a MOS capacitor in the loop filter?</summary>

Its capacitance depends on the gate voltage: 7.48 fF/µm² at 0.9 V, 6.5 at 0.6 V, only 2.2 at 0 V (measured C-V, tt). If V_cont drops below ~0.6 V, C1 collapses, the zero moves up and the PM drops. So the VCO fine-tune range must be centred so V_cont stays in 0.6–1.2 V — a requirement handed to steps 74/75.

</details>

<details><summary><b>Q13.</b> Why is C1 allowed to be a MOS cap but C2 is a MIM cap?</summary>

C1 is the large zero capacitor; a ±19 % variation of it moves the zero and changes PM by only a few degrees. C2 sets the pole and directly takes the charge packet each cycle; its linearity and its absolute value matter more, and it is small (2.2 pF), so a linear MIM costs little area.

</details>

<details><summary><b>Q14.</b> How is R1 realised and why that resistor type?</summary>

res_xhigh_po_2p85 (2.85 µm wide, 701 Ω/µm, 139 Ω heads), L = 15.5 µm → 11.0 kΩ. In the ngspice R/C corners it moves −12 % / +13 % (mostly temperature: tc ≈ −2.5e-3/°C), against −17 % / +20 % for res_high_po. It is compact, and wide enough for low mismatch and flicker. 2.85 µm is one of the five legal widths.

</details>

<details><summary><b>Q15.</b> How much does R1 add to the jitter, and what is the trade-off with β?</summary>

0.31 ps FFNC-off / 0.165 ps FFNC-on (total 2.12 / 1.36 ps). Doubling β halves R (−30 % R noise) but doubles C1 and C2 area; halving β halves area but R-jitter rises to 0.40 ps. Between β = 0.5 and 2 µA/rad the total changes by only 0.03 ps, so β is chosen for area and for a Gm small enough to keep its own noise and offset low.

</details>

<details><summary><b>Q16.</b> Why are the zero and pole at 0.84 MHz and 7.4 MHz?</summary>

They are placed about geometrically around f_c with spread b = tan(45° + lead/2) ≈ 2.9: f_z = f_c/b, f_p = f_c·b (here slightly asymmetric because the sampled-loop correction adds lead). The zero gives the phase lead at f_c; the pole filters the per-cycle charge-packet ripple and the reference spur.

</details>

<details><summary><b>Q17.</b> What is the closed-loop peaking and the 3 dB bandwidth?</summary>

2.5 dB peaking, f_3dB = 4.3 MHz. Peaking amplifies in-band (reference, SSPD, CP) noise near f_c; 2.5 dB is acceptable because those sources are ≥ 10 dB below the VCO contribution in the budget.

</details>

<details><summary><b>Q18.</b> How do you check the loop's lock behaviour at this stage?</summary>

With the same sampled model, cycle by cycle: after a 1 rad reference phase step the error overshoots 27 % and settles inside ±10 ps in 0.34 µs. The 5 µs lock-time spec is therefore set by FLL acquisition (step 81), not by the phase loop.

</details>

<details><summary><b>Q19.</b> Which parameters make up the loop-gain spread m?</summary>

m = (A·g_m·τ·K_VCO)/nominal. A (SSPD amplitude, buffer swing), g_m (bias, mobility), τ (pulser RC) and K_VCO (ring) all vary over PVT; their product can span about ×0.3 to ×2.4 before any trim.

</details>

<details><summary><b>Q20.</b> How robust is the loop to m and to the R/C corners?</summary>

At nominal R/C, PM ≥ 45° for m = 0.46–1.75. At the worst R/C combinations (R −12/+13 %, MIM −12/+13 %, MOS cap −19/+8 % over V_cont 0.6–1.2 V, ss/ff, −40/110 °C) the range narrows to m = 0.78–1.23. With a residual m of 0.75–1.3, PM stays 45.5–50° and f_c 1.8–3.3 MHz at every corner.

</details>

<details><summary><b>Q21.</b> What does icp_trim do and how is it built?</summary>

It changes the number of identical Gm unit cells: 00 → 3 units (nominal), 01 → 4, 10 → 2, 11 → 5, i.e. β ×1, ×1.33, ×0.67, ×1.67. It recentres the process part of m (≈ ×0.55 to ×2.0) to within about ±17 %, leaving room for voltage and temperature drift. Unit cells (rather than a trimmed resistor or pulse width) keep the layout regular and the Gm noise proportional to β.

</details>

<details><summary><b>Q22.</b> Why should code 00 be the nominal setting?</summary>

All controls power up at 0. With 00 = nominal the PLL locks with a sensible bandwidth before any calibration, which makes bring-up and the first measurements safe. The offset-binary decode costs three gates on VDDD.

</details>

<details><summary><b>Q23.</b> What happens if PM drops to 40°?</summary>

The loop is still stable, but peaking rises to about 4 dB, the overshoot grows, and in-band noise is amplified near f_c. The jitter model at PM 40° gives out2 pess 1.84 ps — still inside the 3.2 ps max — so 40° is the floor, 45° the target.

</details>

<details><summary><b>Q24.</b> Why is the continuous approximation good below f_c but wrong near f_REF/2?</summary>

Below f_c the loop sees many samples per cycle of any disturbance, so sampling is invisible. Near f_REF/2 the sampled gain folds and its phase falls faster (delay e^{−jωt_p} plus the hold): the sampled |L| flattens at −25 dB while the continuous one keeps falling. Gardner's rule (f_c < f_REF/10) keeps the difference small; here f_c/f_REF = 1/20.

</details>

<details><summary><b>Q25.</b> What is Gardner's stability limit and how far are we from it?</summary>

For a CP-PLL the sampled loop becomes unstable when the loop gain is so high that the per-cycle correction overshoots; roughly ω_c²·T² limits apply when f_c approaches f_REF/π. At f_c = f_REF/20 we are far below; m = 3 (f_c ≈ 6 MHz) still keeps PM above 30°.

</details>

<details><summary><b>Q26.</b> How does the FLL interact with this loop filter?</summary>

The FLL charge pump injects into the same filter while out of lock. With C_tot ≈ 19 pF a 10 µA FLL current slews V_cont at 0.5 V/µs; once the phase error is inside the dead zone the FLL CP is off and only the pulsed Gm acts. Step 81 sizes the FLL current against this C_tot.

</details>

<details><summary><b>Q27.</b> Why does the filter not use a third pole (4th-order loop)?</summary>

The reference spur from a pulsed-Gm SSPLL is small (no PFD/CP mismatch pulses; in lock the packets are near zero), and C2 with the 7.4 MHz pole already filters the packet ripple at 50 MHz by about 17 dB. An extra RC pole would cost phase margin and area for a spur that is not the limiting spec.

</details>

<details><summary><b>Q28.</b> What leakage matters in this filter?</summary>

Gate leakage of the thin-oxide MOS cap and junction leakage at V_cont. A leakage I_L is supplied by the loop as a static phase offset θ = I_L/β and a small per-cycle ripple I_L·T/C2. For nA-level leakage the offset is mrad and the ripple µV, but it is checked in the loop-filter testbench (step 100) at 110 °C.

</details>

<details><summary><b>Q29.</b> How does temperature move f_c?</summary>

R (xhigh_po) falls about 19 % from −40 to 110 °C (+11 % / −10 % around 27 °C); K_VCO and g_m change too. Since f_c ≈ β·K_VCO·R/2π above the zero, f_c moves with the product; the robustness plot covers this with the R and m axes.

</details>

<details><summary><b>Q30.</b> Why freeze the loop filter now, before the VCO and CP exist?</summary>

C1 and C2 are the largest analog blocks after the CCO and the decap; their area and shape set the floorplan of the CP + LF region. Freezing them (and absorbing K_VCO uncertainty in g_m) lets the floorplan and layout start early.

</details>

<details><summary><b>Q31.</b> What is in the design sheet that the block designers need?</summary>

β·K_VCO product, g_m per unit, τ and t_p limits, R1/C1/C2 with device names and sizes, V_cont window 0.6–1.2 V, icp_trim mapping, and the spread budget for m (process ×0.55–2.0, residual ×0.75–1.3).

</details>

<details><summary><b>Q32.</b> How would you verify the loop parameters in simulation?</summary>

Step 100: AC of the filter impedance at R/C corners (zero, pole). Step 103: open-loop gain from extracted block gains. Step 104: transient lock with a phase/frequency step, compared with the cycle model's 0.34 µs and 27 % overshoot.

</details>

<details><summary><b>Q33.</b> Why does the jitter barely depend on β at fixed f_c and PM?</summary>

Because the only β-dependent noise is R1's, and R1's contribution is small (0.2–0.4 ps against 2.1 ps total). The detector and CP noise referred to VCO phase do not depend on how β is split between g_m and τ, only on A, g_m and d through the budget formulas of step 72.

</details>

<details><summary><b>Q34.</b> Explain the open-loop gain shape in the Bode plot.</summary>

Below the zero: two integrators (VCO and C_tot) → −40 dB/dec and −180°. Between zero and pole: −20 dB/dec, phase rises to a maximum near f_c (the lead). Above the pole: −40 dB/dec again. The sampled version adds extra phase lag from the delay and sampling and flattens near f_REF/2.

</details>

<details><summary><b>Q35.</b> What would change if the reference were 45 or 55 MHz?</summary>

T changes, so the per-cycle charge changes but the average gain β = 2A·g_m·τ/T changes with 1/T: β rises 11 % at 55 MHz and falls 10 % at 45 MHz (m = 1.11 / 0.90) — inside the residual band. f_c/f_REF stays near 1/20.

</details>

<details><summary><b>Q36.</b> Why is the loop parameterised by β and not by an equivalent charge-pump current I_p?</summary>

In a PFD/CP loop the CP current per radian of reference phase is I_p/2π and the divider adds 1/N. In the SSPLL there is no divider and the detector gain is per radian of VCO phase, so β (A/rad of VCO phase) is the natural unit; I_p/(2πN) of a CP-PLL corresponds to β.

</details>

<a id='step-74'></a>
## Step 74 — VCO redesign: stages, LVT, sizing for frequency, K_VCO and phase noise

*Roadmap step 74: schematic and sizing table*

### What happens
The ring's frequency is set by how fast each stage charges its load: more CCO current means faster edges. vtune hangs a
little extra capacitance on every output for fine tuning. Every transistor's noise moves the edges a little. White
(thermal) noise gives phase noise falling at −20 dB/dec. Slow flicker (1/f) noise is up-converted into close-in phase
noise falling at −30 dB/dec, and below the $1/f^3$ corner it dominates.

### How it happens
A current-fed ring settles at the supply voltage $V_{ring}$ where its average current equals the CCO current, so
$f \approx I/(N C_{node} V_{ring})$, $K_I = \partial f/\partial I$, $K_{VCO} = \partial f/\partial v_{tune}$.
Phase noise follows the ISF $h(\tau) = d\phi/dq$:
$$\mathcal{L}_{th}(\Delta f) = \frac{2N\langle h^2 S_{th}\rangle}{2(2\pi\Delta f)^2},\qquad
\mathcal{L}_{fl}(\Delta f) = \frac{2N\langle h\sqrt{B}\rangle^2}{\Delta f\,2(2\pi\Delta f)^2}.$$

### Why it happens this way
Flicker only becomes phase noise through the **dc part** of $h$ (here 0.67 of its rms), and a device's flicker density
scales like $1/(WL)$. The smallest devices on an output node therefore dominate: the fine-tune switch and the coupling TG
nMOS. The main inverters contribute only 13 %. Enlarging just those two devices fixes the noise for 9 % more current;
scaling the whole ring would cost power one-for-one.

**Methods used:** M1 (transient frequency), M2 (ISF with the models' noise), M3 (noise tables), M4 (LVS), M5 (drawing
checks). What, why and how for each: `docs/methods.md`.

**Figure 74.1 — the ISF of one ring node and who causes the flicker.** Hover to compare the base cell and the step-74 cell.

In [ ]:
import json, base64
from IPython.display import Image, display
D = lambda n: json.load(open("../design/vco/data/" + n))
base, q3, ch = D("step74_base.json"), D("step74_q3.json"), D("step74_char_q3.json")

fig = make_subplots(rows=1, cols=2, subplot_titles=("ISF h(τ) of node p1 (rad/pC)", "flicker share (% of the base cell's total)"))
for d, c, n in ((base, "#2a78d6", "base cell"), (q3, "#eb6834", "step-74 cell")):
    h = np.array(d["h"]); ph = np.arange(len(h))/len(h)
    fig.add_trace(go.Scatter(x=np.r_[ph, 1], y=np.r_[h, h[0]]/1e12, name=f"{n} (corner {d['fc3']/1e6:.2f} MHz)",
                             line=dict(color=c)), row=1, col=1)
names = {"x1.xp2": "main pMOS", "x1.xn2": "main nMOS", "x1.xs2": "tune switch S", "x1.xt1n": "TG nMOS (own)",
         "x2.xt1n": "TG nMOS (next)", "x1.xt1p": "TG pMOS (own)", "x2.xt1p": "TG pMOS (next)"}
ref = 10**(base["Lfl"]/10)
for d, c in ((base, "#2a78d6"), (q3, "#eb6834")):
    fig.add_trace(go.Bar(y=list(names.values()), x=[d["shares"][k][1]*10**(d["Lfl"]/10)/ref*100 for k in names],
                         orientation="h", marker_color=c, showlegend=False), row=1, col=2)
fig.update_layout(height=430, barmode="group"); fig.update_xaxes(title_text="injection time / period", row=1, col=1)
show(fig)
print(f"base: Lth {base['Lth']:.2f}, Lfl {base['Lfl']:.2f}, corner {base['fc3']/1e6:.2f} MHz, I {base['I']*1e3:.3f} mA")
print(f"step-74: Lth {q3['Lth']:.2f}, Lfl {q3['Lfl']:.2f}, corner {q3['fc3']/1e6:.2f} MHz, I {q3['I']*1e3:.3f} mA")

**Figure 74.2 — coarse tuning range and ring-node voltage over corners.** Slider: process/temperature corner.

In [ ]:
import json, base64
from IPython.display import Image, display
D = lambda n: json.load(open("../design/vco/data/" + n))
base, q3, ch = D("step74_base.json"), D("step74_q3.json"), D("step74_char_q3.json")

labs = {"tt_27": "tt, 27 °C, 10 fF", "ss_110": "ss, 110 °C, 15 fF", "ff_-40": "ff, −40 °C, 5 fF"}
def frame(k):
    r = ch["f_vs_I"][k]; I = [x[0]*1e3 for x in r]
    return ([dict(x=I, y=[x[1]/1e9 for x in r], name="f (GHz)", line=dict(color="#2a78d6"), row=1, col=1),
             dict(x=I, y=[1.44]*len(I), name="1.44 GHz", line=dict(color="#59636e", dash="dot"), row=1, col=1),
             dict(x=I, y=[1.76]*len(I), name="1.76 GHz", line=dict(color="#59636e", dash="dash"), row=1, col=1),
             dict(x=I, y=[x[2] for x in r], name="ring node (V)", line=dict(color="#eb6834"), row=1, col=2),
             dict(x=I, y=[1.8]*len(I), name="1.8 V limit", line=dict(color="#59636e", dash="dot"), row=1, col=2)],
            labs[k])
def lay(fig):
    fig.update_xaxes(title_text="CCO feed current (mA)")
show(sweep(list(labs), frame, "corner", fmt="{}", active=0, rows=1, cols=2,
           subplot_titles=("frequency (GHz)", "ring supply node (V)"), layout=lay))

**Figure 74.3 — fine tuning.** $K_{VCO}$ is negative; below about 0.7 V the switch turns off and the curve flattens, so the lock window is 0.7–1.2 V.

In [ ]:
import json, base64
from IPython.display import Image, display
D = lambda n: json.load(open("../design/vco/data/" + n))
base, q3, ch = D("step74_base.json"), D("step74_q3.json"), D("step74_char_q3.json")

fig = go.Figure()
cols = {"tt_27": "#2a78d6", "ss_110": "#eb6834", "ff_-40": "#1baf7a"}
for k, c in cols.items():
    r = [x for x in ch["f_vs_vtune"][k] if x[1]]
    fig.add_trace(go.Scatter(x=[x[0] for x in r], y=[x[1]/1e9 for x in r], name=k.replace("_", ", ") + " °C", line=dict(color=c)))
fig.add_vrect(x0=0.7, x1=1.2, fillcolor="#e6e8eb", opacity=0.5, line_width=0, annotation_text="lock window")
fig.update_layout(height=420, xaxis_title="vtune (V)", yaxis_title="f (GHz) at the 1.6 GHz current", title="Fine tuning")
show(fig)

**The schematics** (xschem, hierarchical: cell → ring core → VCO; LVS-clean against the simulated netlist).

In [ ]:
from IPython.display import Image, display
for n in ("sspll_vco_dcell", "sspll_vco_core", "sspll_vco", "sspll_vco_acbuf"):
    print(n); display(Image(filename=f"../docs/img/step74_{n}.png", width=900))

### Sizing table and results
| Device | Model | W × nf | L | Role |
|---|---|---|---|---|
| XP1/XP2 | pfet_01v8 | 3.0 µm × 4 | 0.15 µm | main inverter pMOS |
| XN1/XN2 | nfet_01v8 | 3.0 µm × 2 | 0.15 µm | main inverter nMOS |
| XS1/XS2 | nfet_01v8 | 4.5 µm × 1 | **0.30 µm** | fine-tune switch |
| XC1/XC2 | nfet_01v8 | 4.5 µm × 1 | 0.50 µm | fine-tune MOS cap |
| XT1N/XT2N | nfet_01v8 | 0.84 µm × 1 | **0.30 µm** | coupling TG nMOS |
| XT1P/XT2P | pfet_01v8 | 0.63 µm × 1 | 0.15 µm | coupling TG pMOS |

1.437 mA for 1.6 GHz (tt), ring node 1.45 V; $\mathcal{L}$(1 MHz) −98.7 dBc/Hz thermal, $1/f^3$ corner 0.57 MHz;
$K_{VCO}$ −300 MHz/V (0.6–1.2 V); $K_I$ 0.85 GHz/mA; 1.44–1.76 GHz over corners with the ring node ≤ 1.71 V (10 fF/node);
start-up < 9 ns. System: 2.04 / 1.32 ps (FFNC off/on), 9.55 mW, FoM −227.8 dB. Full report: `docs/design/step74_vco.md`.

### Interview questions — step 74 (31)

<details><summary><b>Q1.</b> Why a ring oscillator and not an LC oscillator for this PLL?</summary>

Area and tuning range. An LC tank at 1.6 GHz needs a nH inductor (hundreds of µm on a side in SKY130's thick metal) and gives a narrow range; a ring is a few hundred µm², tunes over more than ±30 % and is easy to lay out. Its phase noise is ~20 dB worse, which is exactly why this design uses a sub-sampling detector and FFNC: they suppress and cancel ring noise.

</details>

<details><summary><b>Q2.</b> Why 5 stages and not 4?</summary>

A pseudo-differential ring of N stages is two single-ended rings of N inverters coupled by the cross connection. With N = 4 each half is a 4-inverter ring - a non-inverting loop that can latch (two stable states). The proposal's start-up simulations latched at tt, ss and ff. With N = 5 each half is an odd ring and cannot latch; the crossing at the end keeps the differential loop inverting.

</details>

<details><summary><b>Q3.</b> Why not 3 stages?</summary>

The step-6 study: 3 stages run at 2.4 GHz for N = 48 with similar efficiency (VCO FoM within 1 dB) but more absolute jitter at equal power, push the SSPD, NCB and first divider to 2.4–2.6 GHz, and would have changed the submitted proposal. 5 stages at 1.6 GHz were kept.

</details>

<details><summary><b>Q4.</b> What does the always-on transmission gate between the two halves do?</summary>

It ties inp to outp (and inm to outm) so the two single-ended rings run in anti-phase: a weak in-phase coupling that enforces the differential mode. Without it the halves could drift into the common mode. It also adds load and, as the ISF showed, noise - which is why its nMOS was redesigned.

</details>

<details><summary><b>Q5.</b> How is the frequency tuned?</summary>

Coarse: the CCO current into the ring's supply node (5-bit DAC, step 75) - more current, faster edges. Fine: vtune turns on switch S, which hangs a MOS capacitor on each output; more vtune means more capacitance and a lower frequency, so K_VCO is negative (−300 MHz/V average over 0.6–1.2 V). The loop's charge-pump polarity is chosen to match.

</details>

<details><summary><b>Q6.</b> Why is K_VCO negative and does it matter?</summary>

Raising vtune connects more capacitance, slowing the ring. A negative gain is fine as long as the sign of the detector or the charge pump is flipped so the loop has negative feedback. It must be documented, because a wrong sign makes the loop run away to a rail instead of locking.

</details>

<details><summary><b>Q7.</b> What did the ISF reveal about the original cell?</summary>

Thermal noise was dominated by the main pMOS (68 %), but 86 % of the flicker up-conversion came from two small devices: the fine-tune switch (42 %) and the coupling TG nMOS (44 %) - both minimum length (0.15 µm). The main inverter nMOS, normally blamed for flicker, contributed only 4 %.

</details>

<details><summary><b>Q8.</b> Why does a tiny switch dominate the flicker?</summary>

Flicker noise density scales roughly with 1/(W·L): a 0.15 µm device has a tiny gate area, so its 1/f current is large. The switch's on-resistance sets how much of the tuning capacitor the output sees; slow fluctuations of that resistance change the load - that is directly a frequency modulation, i.e. up-converted flicker (the dc part of the ISF).

</details>

<details><summary><b>Q9.</b> How was the 1/f³ corner reduced and what did it cost?</summary>

The switch and the TG nMOS were drawn at L = 0.3 µm (switch W 2.25 → 4.5 µm to keep W/L; TG nMOS 0.63 → 0.84 µm). Corner 1.02 → 0.57 MHz (spec ≤ 0.6). The bigger devices add capacitance: 9 % more current for 1.6 GHz (1.314 → 1.437 mA, ring power 4.86 → 5.33 mW). L = 0.5 µm was tried first and cost 21 % of speed - rejected.

</details>

<details><summary><b>Q10.</b> Why not simply make the main inverters bigger to lower the noise?</summary>

Scaling the whole ring by k lowers all noise by 10·log k but costs k× power - the FoM does not change. The ISF showed the flicker came from two small devices; fixing those costs 9 % power for 2.6 dB lower close-in noise, far better than scaling.

</details>

<details><summary><b>Q11.</b> What is the 1/f³ corner and why does it matter here?</summary>

The offset where up-converted flicker equals thermal phase noise; below it L(f) falls at −30 dB/dec. With the loop at 2.5 MHz, a corner near 1 MHz puts flicker noise right where the loop and FFNC have the least suppression, raising L(300 kHz) FFNC-off above the −110 dBc/Hz spec.

</details>

<details><summary><b>Q12.</b> What does h_dc/h_rms (0.67) tell you?</summary>

How asymmetric the ISF is: the dc part converts slow (flicker) noise into phase drift. A perfectly symmetric waveform would have h_dc = 0 and no flicker up-conversion. 0.67 is high - the rising and falling edges and the load of the tuning branch are not symmetric - so the remaining handle is device area of the noisy devices.

</details>

<details><summary><b>Q13.</b> Why use LVT devices or not?</summary>

pfet_01v8_lvt has a minimum length of 0.35 µm - far too slow for a 1.6 GHz ring. nfet_01v8_lvt would lower the ring-node voltage for a given frequency (more headroom for the CCO), but the standard-Vt ring already runs at 1.45 V at tt and within the limit at ss/110 °C; LVT would add leakage and spread. Kept as a fallback, not used.

</details>

<details><summary><b>Q14.</b> Why is the ring fed by a current and not by a voltage?</summary>

A current-fed ring (CCO) has low supply sensitivity - the high output impedance of the current source isolates it from VDDA noise - and the current directly sets the frequency (K_I ≈ 1.2 GHz/mA). The ring-node voltage settles wherever the ring needs it to draw that current.

</details>

<details><summary><b>Q15.</b> Why must the ring-node voltage stay below 1.8 V?</summary>

The ring uses 1.8 V core devices (absolute maximum 1.95 V) and the CCO output transistor needs headroom from VDDA (2.97 V minimum): every 100 mV more on the ring node costs CCO headroom and raises its noise (step 72).

</details>

<details><summary><b>Q16.</b> Why does every ring node get a buffer, even unused ones?</summary>

Equal loading: each node then has the same capacitance, so all stages have the same delay and the ISF is the same at every node (the 2N-node assumption of the noise calculation holds). It also makes the layout regular - 10 identical buffer cells - and any tap can be used later.

</details>

<details><summary><b>Q17.</b> Why AC-coupled buffers?</summary>

The ring's DC level moves with the CCO current (1.1–1.7 V); a buffer biased by the ring's DC would change its trip point and duty cycle with frequency. The coupling cap removes the DC and the feedback resistor biases the first inverter at its own trip point, so the buffer output is centred at any ring voltage.

</details>

<details><summary><b>Q18.</b> How is the frequency measured in simulation and why that way?</summary>

Transient with an ideal current feed, a differential kick at t = 0, and .meas of 16–20 zero crossings of vop − vom after start-up has settled. A ring has no DC operating point that oscillates, so a small-signal analysis cannot give the frequency; the transient is the simulation equivalent of a frequency counter.

</details>

<details><summary><b>Q19.</b> How was phase noise computed without Spectre PNOISE?</summary>

With the ISF: the charge-injection response h(t) of one node at 12 phases, the bias of every device over a cycle and the PDK's own thermal and flicker noise tables for each device type and length. L = 2N·⟨h²S_th⟩/(2Δω²) for thermal; the flicker part uses ⟨h·√B⟩². It tells not only the total but which device causes it.

</details>

<details><summary><b>Q20.</b> How accurate is the ISF estimate?</summary>

The level is about 6 dB pessimistic against ngspice transient noise (step 7), because the ISF with cyclostationary white noise treats every injection as uncorrelated and the tables use the device's small-signal noise at each instant. The ranking of devices and the 1/f³ corner are reliable. Spectre PNOISE in the Cadence VM is the cross-check (step 95).

</details>

<details><summary><b>Q21.</b> Why use ideal R and C in the buffer for the long characterisation runs?</summary>

The PDK poly resistor is a behavioural (voltage-dependent) model and makes the transient about 5× slower. The ideal replacement uses the PDK values including the resistor's 3 fF end parasitics, and reproduces the PDK-buffer frequency to 0.001 %; the ISF runs and the LVS use the real PDK devices.

</details>

<details><summary><b>Q22.</b> What is K_I and how is it used?</summary>

df/dI of the current-fed ring (Hz/A). It converts the CCO's output current noise into frequency noise (step 72: i_n ≤ 7.2 pA/√Hz) and sets the DAC step size in Hz (step 75).

</details>

<details><summary><b>Q23.</b> How do you check the schematic is the simulated circuit?</summary>

xschem netlists the schematic (LVS format) and netgen compares it with the netlist generated for simulation: all four levels (cell, buffer, core, VCO) match uniquely. The same flow is used for layout-versus-schematic later.

</details>

<details><summary><b>Q24.</b> What layout decisions are already in the schematic?</summary>

Integer finger counts (pMOS 4 × 3 µm, nMOS 2 × 3 µm) so each cell is one abutting row; the two halves are mirror images; the 5 cells are identical; the TGs, switches and caps are drawn per half; 10 identical buffers; MIM bottom plate on the driven ring node, top plate on the high-impedance buffer gate.

</details>

<details><summary><b>Q25.</b> Why is the MIM bottom plate on the ring node?</summary>

The bottom plate (met3) has parasitic capacitance to the substrate. On the strongly driven ring node it only adds a little load; on the high-impedance buffer gate it would form a divider with the coupling cap and attenuate the signal.

</details>

<details><summary><b>Q26.</b> Why is the ring supply node decoupled with only 0.5 pF in simulation?</summary>

The ring is current-fed: a large decap on its supply node would just be charged by the CCO current (it sets the start-up time) and filters the ring's own current ripple. Its value is a layout choice (step 109); 0.5 pF keeps start-up fast.

</details>

<details><summary><b>Q27.</b> How does the frequency change from tt to ss/110 °C, and how is it handled?</summary>

Slower devices and higher temperature lower the frequency at a given current; the CCO DAC supplies more current. The check is that 1.76 GHz is reachable at ss/110 °C with the ring node below the limit, and 1.44 GHz at ff/−40 °C with the DAC's minimum current.

</details>

<details><summary><b>Q28.</b> What sets the start-up time?</summary>

The CCO charging the supply node and decap, then the growth of the differential mode from noise or mismatch. With 5 stages the differential mode is the only unstable mode, so it starts in a few ns at all corners.

</details>

<details><summary><b>Q29.</b> Why 10 fF extra capacitance per node in the simulations?</summary>

A pre-layout allowance for wiring; corners use 5 fF (ff/−40 °C, fast case) and 15 fF (ss/110 °C, slow case). Step 118 replaces it with the extracted values.

</details>

<details><summary><b>Q30.</b> What is the VCO's contribution to the total jitter after this step?</summary>

With the step-74 ring numbers in the step-72 model: 2.04 ps FFNC off, 1.32 ps FFNC on (typ), L(300 kHz) −110.6 / −120.0 dBc/Hz - all inside the typ specs. Power rises to 9.55 mW typ (spec 'about 9'), FoM −227.8 dB unchanged.

</details>

<details><summary><b>Q31.</b> If power had to come back to 9.0 mW, where would you take it?</summary>

The LDO load reserve (0.33 mW), the dummy-buffer size (8 of 10 buffers only provide equal loading - they can be smaller inverters with the same input capacitance), and the NCB bias. Not from the ring: its current buys the noise margin.

</details>